<a href="https://colab.research.google.com/github/kckDeepak/Sentiment-Analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


Imports

In [4]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

Load IMDB Dataset

In [5]:
from datasets import load_dataset

dataset = load_dataset("imdb")   # This is the standard one (stanfordnlp/imdb)
print(dataset)  # Shows train, test, unsupervised splits

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Using only a Small Subset

In [6]:
# For fast testing (remove these 3 lines when you want full training)
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))
dataset = dataset.map(lambda x: x, remove_columns=dataset["train"].column_names)  # dummy
dataset["train"] = small_train
dataset["test"] = small_test

Now We Tokenize the Text

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)  # dynamic padding later

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading the Pretrained Model

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluation Metrics

In [9]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

Training Arguments

In [10]:
training_args = TrainingArguments(
    output_dir="imdb-sentiment-model",
    eval_strategy="epoch",          # evaluate after each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,             # 3 epochs is usually enough
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,              # set True later if you want to share
    report_to="none",               # no wandb/tensorboard spam
)

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Train the Model

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.313320,0.864000
2,No log,0.359411,0.860000
3,No log,0.330945,0.890000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=375, training_loss=0.2746822916666667, metrics={'train_runtime': 355.0921, 'train_samples_per_second': 16.897, 'train_steps_per_second': 1.056, 'total_flos': 783230052978432.0, 'train_loss': 0.2746822916666667, 'epoch': 3.0})

Save the Model Locally

In [21]:
trainer.save_model("my_imdb_sentiment_model")
tokenizer.save_pretrained("my_imdb_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_imdb_sentiment_model/tokenizer_config.json',
 'my_imdb_sentiment_model/tokenizer.json')

Evaluate on Test Set

In [28]:
# ============================
# TESTING / INFERENCE - SINGLE CELL
# ============================

from transformers import pipeline
import torch

# Load your fine-tuned model for inference
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="my_imdb_sentiment_model",
    tokenizer="my_imdb_sentiment_model",
    device=0 if torch.cuda.is_available() else -1,   # Use GPU if available
    truncation=True,
    max_length=512
)

# Test examples
test_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Worst film I've ever seen. Complete waste of time and money.",
    "It was okay, not great but not terrible either.",
    "The acting was superb but the plot was confusing and too long.",
    "One of the best movies of the year! Highly recommended.",
    "I fell asleep halfway through. Very boring.",
    "The visuals were stunning but the story was weak.",
    "A masterpiece! Best movie I've watched in a long time."
]

print("=== SENTIMENT ANALYSIS TEST RESULTS ===\n")

for i, review in enumerate(test_reviews, 1):
    result = sentiment_pipeline(review)[0]

    # Convert LABEL_0 / LABEL_1 to readable text
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    confidence = result["score"] * 100

    print(f"{i}. Review: {review}")
    print(f"   Prediction: {label} ({confidence:.2f}% confidence)")
    print("-" * 80)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== SENTIMENT ANALYSIS TEST RESULTS ===

1. Review: This movie was absolutely fantastic! I loved every minute of it.
   Prediction: POSITIVE (91.68% confidence)
--------------------------------------------------------------------------------
2. Review: Worst film I've ever seen. Complete waste of time and money.
   Prediction: NEGATIVE (92.45% confidence)
--------------------------------------------------------------------------------
3. Review: It was okay, not great but not terrible either.
   Prediction: NEGATIVE (52.86% confidence)
--------------------------------------------------------------------------------
4. Review: The acting was superb but the plot was confusing and too long.
   Prediction: NEGATIVE (86.65% confidence)
--------------------------------------------------------------------------------
5. Review: One of the best movies of the year! Highly recommended.
   Prediction: POSITIVE (87.79% confidence)
-------------------------------------------------------------------

Test Model on New Reviews

In [32]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="my_imdb_sentiment_model",
    tokenizer=tokenizer
)

examples = [
    "I absolutely loved this movie! The acting was brilliant.",
    "This was the worst film I've ever seen. Total waste of time.",
    "It was okay, nothing special but not terrible either.",
    "This is cinema!",
    "Life changing experience",
    "Piece of crap"
]

for ex in examples:
    result = sentiment_analyzer(ex)[0]
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    print(f"Review: {ex}\nSentiment: {label} (score: {result['score']:.4f})\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Review: I absolutely loved this movie! The acting was brilliant.
Sentiment: POSITIVE (score: 0.9259)

Review: This was the worst film I've ever seen. Total waste of time.
Sentiment: NEGATIVE (score: 0.9270)

Review: It was okay, nothing special but not terrible either.
Sentiment: NEGATIVE (score: 0.6374)

Review: This is cinema!
Sentiment: POSITIVE (score: 0.7468)

Review: Life changing experience
Sentiment: POSITIVE (score: 0.7869)

Review: Piece of crap
Sentiment: NEGATIVE (score: 0.8654)

